# Notebook 006 — Enriquecimento Funcional por Módulo (ORA)

Over-representation analysis (ORA) dos 6 módulos WGCNA selecionados.
Para cada módulo, testa se os genes DEG de prioridade estão sobre-representados
em termos GO (Biological Process) e vias KEGG, usando o fundo de 12.176 genes
proteína-codificantes medidos no experimento como referência.

**Requer:** `pip install gseapy` e conexão com a internet (API do Enrichr).

**Lógica de seleção de genes:**
- Fundo: todos os 12.176 genes proteína-codificantes filtrados (notebook 001)
- Lista por módulo: genes que são DEG de prioridade **e** pertencem ao módulo
  - DEG de prioridade: padj < 0.10, |log2FC| ≥ 1.0, em qualquer contraste prioritário
  - Contraste prioritário: MA1_vs_CTR, MA100_vs_CTR, MC1_vs_CTR, MD1_vs_CTR, MA100_vs_MA1, MC100_vs_MC1

**Referência metodológica:** Sainz et al. (2024) — etapa de enriquecimento GO/KEGG por módulo.

## Imports

In [ ]:
import gseapy as gp
from pathlib import Path

import pandas as pd
import numpy as np
import plotly.express as px

## Configurações

In [ ]:
ANNOTATION_DIR = Path("../../data/interim/annotation")
NETWORK_DIR    = Path("../../data/processed/network_export")
ENRICHMENT_DIR = Path("../../data/processed/enrichment")
ENRICHMENT_DIR.mkdir(parents=True, exist_ok=True)

MODULES_ANN_PATH = ANNOTATION_DIR / "wgcna_gene_modules_annotated.csv"
DEG_ANN_PATH     = ANNOTATION_DIR / "deg_all_contrasts_annotated.csv"
MODULES_SEL_PATH = NETWORK_DIR    / "selected_modules_summary.csv"

# Contrastes de prioridade — mesmos usados no notebook 005
PRIORITY_CONTRASTS = [
    "MA1_vs_CTR", "MA100_vs_CTR", "MC1_vs_CTR",
    "MD1_vs_CTR", "MA100_vs_MA1", "MC100_vs_MC1",
]

# Thresholds de DEG — mesmos usados no notebook 005
PADJ_THRESHOLD = 0.10
LFC_THRESHOLD  = 1.0

# Bibliotecas de gene sets consultadas via API do Enrichr
GENE_SETS = ["GO_Biological_Process_2023", "KEGG_2021_Human"]

## Carregamento dos Dados

In [ ]:
modules_ann = pd.read_csv(MODULES_ANN_PATH)
deg_ann     = pd.read_csv(DEG_ANN_PATH)
modules_sel = pd.read_csv(MODULES_SEL_PATH)

selected_modules = modules_sel["module"].tolist()

print("Módulos selecionados:", selected_modules)
print("Genes anotados (fundo):", modules_ann.shape[0])
print("Linhas DEG anotadas:", deg_ann.shape[0])

display(modules_ann.head(3))
display(deg_ann.head(3))

## Construção das Listas de Genes por Módulo

In [ ]:
# Fundo: todos os símbolos HGNC dos 12.176 genes medidos
background_symbols = modules_ann["gene_symbol"].dropna().unique().tolist()
print(f"Fundo: {len(background_symbols)} genes")

# DEGs de prioridade: contraste prioritário + padj < 0.10 + |log2FC| >= 1.0
priority_degs = deg_ann[
    (deg_ann["contrast"].isin(PRIORITY_CONTRASTS)) &
    (deg_ann["padj"] < PADJ_THRESHOLD) &
    (deg_ann["log2FoldChange"].abs() >= LFC_THRESHOLD)
].copy()

# Adiciona coluna de módulo
priority_degs = priority_degs.merge(
    modules_ann[["gene_id", "module"]],
    on="gene_id",
    how="inner"
)

# Genes únicos por módulo (um gene pode aparecer em vários contrasts)
module_gene_lists = {}
for module in selected_modules:
    genes = (
        priority_degs[priority_degs["module"] == module]
        ["gene_symbol"]
        .dropna()
        .unique()
        .tolist()
    )
    module_gene_lists[module] = genes

print("\nGenes por módulo (entrada para ORA):")
for mod, genes in module_gene_lists.items():
    note = " ⚠ lista pequena" if len(genes) < 15 else ""
    print(f"  {mod}: {len(genes)} genes{note}")

## Enriquecimento Funcional (ORA via Enrichr API)

Para cada módulo com ≥ 5 genes DEG de prioridade, executa ORA usando o Enrichr.
O fundo customizado (12.176 genes) garante que a sobre-representação seja calculada
em relação apenas aos genes que puderam ser detectados no experimento.

In [ ]:
all_results = []

for module, gene_list in module_gene_lists.items():
    print(f"\n{'='*55}")
    print(f"Módulo: {module}  |  {len(gene_list)} genes DEG de prioridade")

    if len(gene_list) < 5:
        print("  ⚠ Menos de 5 genes — ORA não confiável. Pulando.")
        continue

    if len(gene_list) < 15:
        print("  ⚠ Lista pequena — ORA executada mas resultados podem não ser significativos.")

    try:
        enr = gp.enrichr(
            gene_list=gene_list,
            gene_sets=GENE_SETS,
            background=background_symbols,
            organism="human",
            outdir=None,
            verbose=False,
        )
        df = enr.results.copy()
        df["module"] = module
        df["n_genes_input"] = len(gene_list)

        n_sig = (df["Adjusted P-value"] < 0.05).sum()
        print(f"  Termos significativos (FDR < 0.05): {n_sig}")

        all_results.append(df)

        out_path = ENRICHMENT_DIR / f"{module}_enrichr_results.csv"
        df.to_csv(out_path, index=False)
        print(f"  Salvo: {out_path.name}")

    except Exception as e:
        print(f"  ERRO: {e}")

## Consolidação dos Resultados

In [ ]:
if not all_results:
    print("Nenhum resultado de enriquecimento foi gerado.")
else:
    combined     = pd.concat(all_results, ignore_index=True)
    sig_combined = combined[combined["Adjusted P-value"] < 0.05].copy()

    print(f"Total de termos significativos (FDR < 0.05): {len(sig_combined)}")
    print("\nDistribuição por módulo e gene set:")
    display(
        sig_combined.groupby(["module", "Gene_set"])
        .size()
        .reset_index(name="n_termos_sig")
    )

    # Tabela resumo: top 5 GO BP por módulo
    top5_bp = (
        sig_combined[sig_combined["Gene_set"] == "GO_Biological_Process_2023"]
        .sort_values(["module", "Adjusted P-value"])
        .groupby("module")
        .head(5)
        [["module", "Term", "Adjusted P-value", "Genes"]]
        .reset_index(drop=True)
    )

    summary_path = ENRICHMENT_DIR / "enrichment_summary.csv"
    top5_bp.to_csv(summary_path, index=False)
    print(f"\nResumo (top 5 GO BP por módulo) salvo em: {summary_path.name}")
    display(top5_bp)

## Visualização — Dotplots por Módulo

In [ ]:
def dotplot_module(df_sig, module_name, gene_set, top_n=10):
    df = (
        df_sig[df_sig["Gene_set"] == gene_set]
        .sort_values("Adjusted P-value")
        .head(top_n)
        .copy()
    )
    if df.empty:
        return None

    df["-log10(padj)"] = -np.log10(df["Adjusted P-value"].clip(lower=1e-300))
    df["n_overlap"]    = df["Genes"].apply(
        lambda x: len(str(x).split(";")) if pd.notna(x) and str(x).strip() else 0
    )
    df["Term_label"]   = df["Term"].apply(lambda t: (t[:60] + "…") if len(t) > 62 else t)

    fig = px.scatter(
        df,
        x="-log10(padj)",
        y="Term_label",
        size="n_overlap",
        color="Adjusted P-value",
        color_continuous_scale="RdBu",
        labels={
            "-log10(padj)": "-log₁₀(padj)",
            "Term_label":    "",
            "n_overlap":     "Genes (overlap)",
            "Adjusted P-value": "padj",
        },
        title=f"{module_name} — {gene_set} (top {top_n})",
    )
    fig.update_layout(
        yaxis=dict(autorange="reversed"),
        height=max(300, 32 * len(df) + 120),
        width=850,
        margin=dict(l=320, r=60, t=60, b=40),
    )
    return fig


for module in selected_modules:
    result_path = ENRICHMENT_DIR / f"{module}_enrichr_results.csv"
    if not result_path.exists():
        continue

    df_mod = pd.read_csv(result_path)
    df_sig = df_mod[df_mod["Adjusted P-value"] < 0.05]

    if df_sig.empty:
        print(f"[{module}] Sem termos significativos — nenhum dotplot gerado.")
        continue

    for gs in GENE_SETS:
        fig = dotplot_module(df_sig, module, gs)
        if fig is not None:
            fig.show()

## Resumo Final

In [ ]:
print("=" * 60)
print("RESUMO DO ENRIQUECIMENTO POR MÓDULO")
print("=" * 60)

for module, gene_list in module_gene_lists.items():
    result_path = ENRICHMENT_DIR / f"{module}_enrichr_results.csv"
    print(f"\n{module} ({len(gene_list)} genes DEG de prioridade):")

    if not result_path.exists():
        if len(gene_list) < 5:
            print("  → Menos de 5 genes; ORA não executada.")
        else:
            print("  → Arquivo de resultado não encontrado (verifique erros acima).")
        continue

    df_mod = pd.read_csv(result_path)
    df_sig = df_mod[df_mod["Adjusted P-value"] < 0.05]

    if df_sig.empty:
        print("  → Nenhum termo com FDR < 0.05.")
        continue

    for gs in GENE_SETS:
        sub = df_sig[df_sig["Gene_set"] == gs].sort_values("Adjusted P-value")
        if sub.empty:
            continue
        top1 = sub.iloc[0]
        n    = len(sub)
        print(f"  {gs}: {n} termos sig. | top → {top1['Term']} (padj={top1['Adjusted P-value']:.3e})")